In [1]:
import os
from pathlib import Path
import sys
import pandas as pd

In [2]:
# Add the scripts folder to the system path
script_dir = Path("../scripts").resolve()
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

In [3]:
from VT3_utils import extract_date_from_filename, load_mplnet_data, creating_mask

## MPLNET

In [4]:
def process_mplnet_directory(parent_directory):
    parent_directory = Path(parent_directory)
    print(f'Processing MPLNET data files')

    # Initialize all final containers
    final_time = []
    final_extinction = []
    final_backscatter = []
    final_depol = []
    final_aod = []
    final_mask = []
    flags_b_e, neg_samples, nan_b_e, depol_1, flags_depol, nan_depol = [], [], [], [], [], []

    flag = 0  # Used to set metadata once

    # Define subdirectories inside the main folder
    backscatter_dir = parent_directory / 'mplnet_backscatter'
    extinction_dir = parent_directory / 'mplnet_extinction'
    depol_dir = parent_directory / 'mplnet_depol'
    aod_dir = parent_directory / 'mplnet_aod'

    # Sorted list of files
    backscatter_files = sorted(backscatter_dir.glob('*.nc4'))
    extinction_files = sorted(extinction_dir.glob('*.nc4'))
    depol_files = sorted(depol_dir.glob('*.nc4'))
    aod_files = sorted(aod_dir.glob('*.nc4'))

    for b_file, e_file, d_file, aod_file in zip(backscatter_files, extinction_files, depol_files, aod_files):
        # ➤ Custom Function: extract_date_from_filename
        b_date = extract_date_from_filename(b_file)
        e_date = extract_date_from_filename(e_file)
        d_date = extract_date_from_filename(d_file)
        aod_date = extract_date_from_filename(aod_file)

        if b_date == e_date == d_date == aod_date:
            dataset_extinction = nc.Dataset(e_file, mode='r') 
            dataset_depol = nc.Dataset(d_file, mode='r')
            dataset_backscatter = nc.Dataset(b_file, mode='r')
            dataset_aod = nc.Dataset(aod_file, mode='r')
            file_paths = [e_file, d_file, b_file, None, aod_file]

            # ➤ Custom Function: load_mplnet_data
            data = load_mplnet_data(file_paths)
            total_extinction = data['total_extinction']
            total_backscatter = data['total_backscatter']
            total_depol = data['total_depol']
            total_aod = data['total_aod']
            time = data['time']

            # ➤ Custom Function: creating_mask
            mm, flag_mask_back, mask_neg, mask_back_and_ext, flag_mask_depol, depol_greater_1, nan_mask_depol = creating_mask(
                dataset_extinction, dataset_backscatter, dataset_depol
            )

            # Capture metadata only once
            if flag == 0:
                altitudes = data['altitudes']
                altitudes_meter = altitudes * 1000
                latitude = data['latitude']
                longitude = data['longitude']
                flag = 1

            # Append all valid data
            final_time.extend(time)
            final_extinction.append(total_extinction)
            final_backscatter.append(total_backscatter)
            final_depol.append(total_depol)
            final_aod.append(total_aod)
            final_mask.append(mm)
            flags_b_e.append(flag_mask_back)
            neg_samples.append(mask_neg)
            nan_b_e.append(mask_back_and_ext)
            depol_1.append(depol_greater_1)
            flags_depol.append(flag_mask_depol)
            nan_depol.append(nan_mask_depol)

    # Stack everything after loop
    final_time_array = np.array(final_time)
    final_extinction = np.ma.concatenate(final_extinction, axis=0)
    final_backscatter = np.ma.concatenate(final_backscatter, axis=0)
    final_depol = np.ma.concatenate(final_depol, axis=0)
    final_aod = np.ma.concatenate(final_aod, axis=0)
    final_mask = np.ma.concatenate(final_mask, axis=0)
    flags_b_e = np.ma.concatenate(flags_b_e, axis=0)
    neg_samples = np.ma.concatenate(neg_samples, axis=0)
    nan_b_e = np.ma.concatenate(nan_b_e, axis=0)
    depol_1 = np.ma.concatenate(depol_1, axis=0)
    flags_depol = np.ma.concatenate(flags_depol, axis=0)
    nan_depol = np.ma.concatenate(nan_depol, axis=0)

    # Apply masks to filter data
    extinction_filtered = np.ma.array(final_extinction, mask=final_mask)
    backscatter_filtered = np.ma.array(final_backscatter, mask=final_mask)
    depol_filtered = np.ma.array(final_depol, mask=final_mask)

    print(f'Processed all data files. Storing in a dictionary.')

    # Return as dictionary
    return {
        "time": final_time_array,
        "extinction": final_extinction,
        "backscatter": final_backscatter,
        "depol": final_depol,
        "aod": final_aod,
        "mask": final_mask,
        "extinction_filtered": extinction_filtered,
        "backscatter_filtered": backscatter_filtered,
        "depol_filtered": depol_filtered,
        "altitudes": altitudes_meter,
        "latitude": latitude,
        "longitude": longitude
    }


In [5]:
def select_MT_samples(parameter, final_time, avg_range):
    """
    It evaluates the average dust extinction every 3 hours. For each time
    sample (00, 03, 06, 09, 12, 15, 18, 21) it averages avg_range samples before and after.

    Parameters:
    -----------

    parameter : numpy.ndarray
        Parameter to be averaged.
    final_time: list datetime objects
        The time variable of the parameter converted into a np array.
    avg_range : int
        The range of minutes I want to compute the average.

    Returns:
    --------

    parameter_monarch_time : numpy.ndarray
        Parameter averaged every 3 hours.

    """
    n = 180
    final_time_array = np.array(final_time)
    start_time = final_time[0]  # This should be a datetime object

    time_3hourly = [start_time + timedelta(hours=3) * i for i in range((len(final_time) // n))]

    parameter_monarch_time = []

    for t in time_3hourly: 
        closest_idx = np.argmin(np.abs(final_time_array - t))
        # print(final_time[closest_idx])
        start_idx = max(closest_idx - avg_range, 0)
        # print(start_idx)
        end_idx = min(closest_idx + avg_range, len(final_time))
        # print(end_idx)
        
        if parameter.ndim == 1:
            de_avg = np.ma.mean(parameter[start_idx:end_idx])
        else: 
            de_avg = np.ma.mean(parameter[start_idx:end_idx,:], axis = 0)
            
        parameter_monarch_time.append(de_avg)

    parameter_monarch_time = np.ma.array(parameter_monarch_time)
    
    return parameter_monarch_time, time_3hourly

In [6]:
def save_to_csv(time, parameter, parameter_name, directory_path, altitudes=None):
    """
    Saves model data to a CSV file. Supports both multi-level (with altitude) and single-level data.

    Parameters:
    ----------
    time : array-like
        List or array of datetime values.

    parameter : array-like
        Data array to be saved. For multi-level data, should be 2D (time x levels).

    parameter_name : str
        Name used to construct the output CSV file name.

    directory_path : str or Path
        Path to the directory where the CSV will be saved.

    altitudes : array-like, optional
        List of altitudes corresponding to vertical levels. If provided, used as column headers.
    """
    os.makedirs(directory_path, exist_ok=True)

    if altitudes is not None:
        if len(parameter) != len(time):
            print(f"Error: parameter and time arrays have mismatched lengths.")
            return

        altitude_columns = [f"{alt:.2f}" for alt in altitudes]
        df_parameter = pd.DataFrame(parameter, columns=altitude_columns)
    else:
        df_parameter = pd.DataFrame(parameter)

    df_parameter.insert(0, "time", time)
    
    file_name = f"{parameter_name}.csv"
    output_file_path = os.path.join(directory_path, file_name)
    df_parameter.to_csv(output_file_path, index=False)
    print(f"CSV file saved to {output_file_path}")


## MONARCH

In [7]:
def process_monarch_variable(variable_name, sites, directory_path, output_base_path):
    """
    Processes NetCDF MONARCH output files to extract time series data for a specified variable
    at given geographic sites, and saves the results as CSV files.

    Parameters:
    ----------
    variable_name : str
        The name of the variable to extract. Must be one of:
        - 'sconc_dust'       : Dust concentration across vertical levels.
        - 'dust_load'    : Column-integrated dust mass loading.
        - 'od550_dust'   : Dust aerosol optical depth at 550 nm.

    sites : dict
        Dictionary of sites with site names as keys and their latitude and longitude as values.
        Example:
            {
                "SiteA": {"latitude": 35.5, "longitude": -5.5},
                "SiteB": {"latitude": 32.0, "longitude": -4.2}
            }

    directory_path : pathlib.Path
        Path to the directory containing NetCDF (*.nc) MONARCH files.

    output_base_path : pathlib.Path
        Path to the base directory where CSV outputs will be saved.

    """
    file_paths = sorted(directory_path.glob("*.nc"))

    for site_name, coords in sites.items():
        print(f"\nProcessing site: {site_name}")
        site_latitude = coords["latitude"]
        site_longitude = coords["longitude"]

        site_directory = Path(output_base_path)
        site_directory.mkdir(parents=True, exist_ok=True)

        final_time = []
        data_collector = []
        levels = None

        for file_path in file_paths:
            file_name = os.path.basename(file_path)
            print("Processing file:", file_name)

            dataset = nc.Dataset(file_path, mode="r")

            # Locate closest grid point
            latitude = dataset.variables["lat"][:]
            longitude = dataset.variables["lon"][:]

            if "rlat" in dataset.dimensions and "rlon" in dataset.dimensions:
                combined_diff = np.sqrt((latitude - site_latitude) ** 2 + (longitude - site_longitude) ** 2)
                lat_row, lon_col = np.unravel_index(combined_diff.argmin(), combined_diff.shape)
            else:
                lat_row = (np.abs(latitude - site_latitude)).argmin()
                lon_col = (np.abs(longitude - site_longitude)).argmin()

            # Extract time and convert
            time_monarch = dataset.variables["time"][4:13]
            file_day = str(file_name).split("_")[2][:8] # modify the split in case of more underscores
            reference_time = datetime.strptime(file_day, "%Y%m%d") + timedelta(hours=12)
            time_monarch_datetime = [reference_time + timedelta(hours=float(t)) for t in time_monarch]
            final_time.extend(time_monarch_datetime)

            # Extract and store variable
            if variable_name == "sconc_dust":
                if levels is None:
                    levels = dataset.variables["lev"][:]
                data = dataset.variables["sconc_dust"][4:13, :, lat_row, lon_col]
                data_collector.append(data)

            elif variable_name == "dust_load":
                data = dataset.variables["dust_load"][4:13, lat_row, lon_col]
                data_collector.append(data)

            elif variable_name == "od550_dust":
                data = dataset.variables["od550_dust"][4:13, lat_row, lon_col]
                data_collector.append(data)

            else:
                print(f"Unsupported variable: {variable_name}")
                continue

            dataset.close()

        # Concatenate and save
        final_time_array = np.array(final_time)
        data_array = np.ma.concatenate(data_collector, axis=0)
        output_filename = f"{variable_name}_{site_name}"

        if variable_name == "sconc_dust":
            save_to_csv(final_time_array, data_array, output_filename, site_directory, levels)
        else:
            save_to_csv(final_time_array, data_array, output_filename, site_directory)
            


## Plotting MPLNET vs MONARCH

In [8]:
def select_timesteps_7_9(ds):
    """
    Returns a slice of the dataset between index 7 and 9, equivalent to 09UTC and 12UTC.
    """
    return ds.isel(time=slice(7, 9))

## VIIRS vs MONARCH comparison

In [9]:
def daily_average(parameter_df, avg_type):
    # Create a true copy of the DataFrame to avoid modifying the original
    parameter_copy = parameter_df.copy()
    
    parameter_copy['time'] = pd.to_datetime(parameter_copy['time'])
    parameter_copy.set_index('time', inplace=True)
    
    if avg_type == 'from 3 to 21':
        filtered_parameter_df = parameter_copy.between_time("03:00", "21:01")
        daily_average_df = filtered_parameter_df.resample('D').mean() 
        
    elif avg_type == 'all day':
        daily_average_df = parameter_copy.resample('D').mean()
        
    return daily_average_df

In [10]:
def diff_normalise(levels):
    """
    Return a BoundaryNorm object for custom colorbar levels.

    Parameters:
    - levels: list of numeric boundaries (e.g., [-5, -2, 0, 2, 5])

    Returns:
    - norm: BoundaryNorm object
    """
    return BoundaryNorm(boundaries=levels, ncolors=256, clip=False, extend='both')

In [11]:
def comparison_plot(ax, data, lat_min, lat_max, lon_min, lon_max, plot_type="DOD"):
    """
    Plots a single panel on provided ax.

    Parameters:
    - ax: Matplotlib axis with PlateCarree projection.
    - data: xarray.DataArray to plot.
    - lat/lon min/max: float, geographic extent.
    - plot_type: "variable" or "diff".
    """

    if plot_type == "DOD":
        dod_norm = [0, 0.1, 0.2, 0.4, 0.8, 1.2]
        cmap = "viridis"
        norm = get_custom_norm(dod_norm)
        cbar_label = "DOD"
    elif plot_type == "diff":
        diff_norm = [-0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8]
        cmap = "RdBu_r"
        max_abs = np.nanmax(np.abs(data.values))
        norm = diff_normalise(diff_norm)
        #norm = mcolors.TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
        cbar_label = "DOD difference"
    else:
        raise ValueError("plot_type must be 'DOD' or 'diff'")

    # Plotting
    p = ax.pcolormesh(data["lon"], data["lat"], data.values.squeeze(),
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm,
                      shading="auto")
    
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.coastlines()
    
    # Colorbar for this panel
    cbar = plt.colorbar(p, ax=ax, orientation="vertical", label=cbar_label, shrink=0.8)

    return ax  # Optional, for chaining